# 04 · E2 (ajuste fino) y E3 (segunda arquitectura)
**Entrega 2 · días 8–10.** Necesita GPU y que E1 ya esté ejecutado. Las hipótesis de E2 y E3 deben estar escritas antes.

In [ ]:
# PREPARACIÓN (igual en todos los notebooks): conecta Drive, entra a la carpeta del proyecto y deja listos los datos.
from google.colab import drive
drive.mount('/content/drive')

PROYECTO = '/content/drive/MyDrive/yuca-comparacion'   # carpeta del proyecto dentro de su Google Drive

import os, sys
os.chdir(PROYECTO)
sys.path.insert(0, PROYECTO)

from src.datos import descargar_dataset
# El .zip se guarda una sola vez en Drive; las fotos se descomprimen en el disco de Colab (más rápido).
RAIZ = str(descargar_dataset(destino='/content/datos', carpeta_zip=PROYECTO + '/data'))
os.environ['YUCA_DATOS'] = RAIZ
print('Proyecto:', os.getcwd())
print('Datos   :', RAIZ)

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU: menú Entorno de ejecución > Cambiar tipo de entorno > T4 GPU')

## E3 · Backbone B congelado
Mismo procedimiento, mismas épocas y misma tasa de aprendizaje que E1; solo cambia la arquitectura.
**Comparación primaria: E1 frente a E3.**

In [ ]:
from src.entrenar import ejecutar
from IPython.display import Image as Mostrar, display
import pandas as pd
m3 = ejecutar('E3')

In [ ]:
display(Mostrar('resultados/E3/curvas.png'))
display(Mostrar('resultados/E3/matriz_confusion.png'))
pd.DataFrame(m3['por_clase']).T

## E2 · Ajuste fino del Backbone A
Parte de los pesos del mejor E1, descongela el último bloque del backbone y entrena con una tasa de aprendizaje 10 veces menor.
**Comparación secundaria: E1 frente a E2.** El costo de E2 incluye el de E1 (tiempo acumulado).

In [ ]:
m2 = ejecutar('E2')

In [ ]:
display(Mostrar('resultados/E2/curvas.png'))
display(Mostrar('resultados/E2/matriz_confusion.png'))
pd.DataFrame(m2['por_clase']).T

## Tabla consolidada E0–E3

In [ ]:
from src import analisis
consolidada = analisis.graficas_comparacion()
consolidada.T

In [ ]:
display(Mostrar('resultados/comparacion_f1_por_clase.png'))
display(Mostrar('resultados/comparacion_costo.png'))
analisis.tabla_por_clase(metrica='recall')

## Estabilidad (opcional, recomendado)
Repite E1 y E3 con otras semillas para ver cuánto cambian los resultados por azar.
Si la diferencia entre E1 y E3 es menor que esa variación, no se puede afirmar que una arquitectura sea mejor.

In [ ]:
SEMILLAS_EXTRA = [1, 2]   # decisión del equipo; cada una repite E1 y E3 completos
for s in SEMILLAS_EXTRA:
    ejecutar('E1', semilla=s)
    ejecutar('E3', semilla=s)
detalle, resumen = analisis.estabilidad()
display(detalle)
resumen